# reidfm: evaluate on every test split + OOD (Kaggle, 1 GPU is enough)

Attach the pool datasets and the training notebook's output. Produces `results/<name>.json`
and `.md` in the benchmark protocol, plus an exported checkpoint that the benchmark's
`FastReIDEncoder` loads directly.

In [ ]:
import os
# ---- edit these ----
REPO_URL = "https://github.com/<you>/Person-Re-ID-Foundational-model.git"
RUN_DIR = "/kaggle/input/reidfm-train-s2/output/sbs_r50ibn_msmt_v1"   # a finished run folder
EXPORT_KEY = "reidfm_sbs_r50_ibn_v1"
BASELINES = ["sbs_r50_ibn_msmt", "osnet_ain_msdc"]   # zero-shot E0 rows; [] to skip
SETS = "all"      # val | test | ood | all | comma-separated keys
# --------------------
os.environ["REIDFM_DATA_ROOT"] = "/kaggle/input"
os.environ["REIDFM_WEIGHTS_ROOT"] = "/kaggle/working/weights"

In [ ]:
%cd /kaggle/working
!test -d reidfm-repo || git clone -q $REPO_URL reidfm-repo
%cd /kaggle/working/reidfm-repo
!git pull -q
!python scripts/setup_fastreid.py --cython
!pip install -q -e ".[train]"

In [ ]:
# The trained model: best checkpoint by the mean validation metric.
!python scripts/evaluate.py --checkpoint $RUN_DIR/model_best.pth --sets $SETS --skip-missing --name $EXPORT_KEY

In [ ]:
# Zero-shot starting points on the same sets, for the comparison table.
for key in BASELINES:
    !python scripts/evaluate.py --model $key --sets $SETS --skip-missing

In [ ]:
# Export for the benchmark / downstream use, and show all result tables.
!python scripts/export_weights.py --run $RUN_DIR --key $EXPORT_KEY --out-dir /kaggle/working/export --register
!cp configs/models/exported/$EXPORT_KEY.yml /kaggle/working/export/
!mkdir -p /kaggle/working/results && cp results/*.json results/*.md /kaggle/working/results/
!cat /kaggle/working/results/*.md